# LoRA Fine-Tuning — DeBERTa-v3-large para NLI Jurídico 5 Clases

**Modelo base:** `cross-encoder/nli-deberta-v3-large` (~450M parámetros)

**Tarea:** Clasificación de 5 relaciones normativas:
- 0: CONTRADICCIÓN
- 1: COMPATIBILIDAD
- 2: ESPECIFICACIÓN
- 3: EXCEPCIÓN
- 4: NO_RELACIÓN

**Por qué LoRA y no full fine-tuning:**
Con 2500 muestras (500 por clase) y 450M parámetros, full fine-tuning
sobreajusta. LoRA entrena solo ~5-8M parámetros adicionales, preservando
el conocimiento del preentrenamiento y generalizando mejor con pocos datos.

In [1]:
# Verificar GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# Instalar dependencias
!pip install transformers[torch] datasets peft scikit-learn \
             huggingface_hub accelerate -q
print("Instaladas")

In [4]:
# Cargar y preparar dataset
import json, random
import numpy as np
from collections import Counter

DATASET_PATH = "dataset_5clases.json"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Mapeo de etiquetas
ID2LABEL = {
    0: "CONTRADICCION",
    1: "COMPATIBILIDAD",
    2: "ESPECIFICACION",
    3: "EXCEPCION",
    4: "NO_RELACION",
}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = 5

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)["dataset"]

# Normalizar estructura
datos = []
for item in raw:
    label = item.get("label")
    if isinstance(label, str):
        label = LABEL2ID.get(label.upper(), 4)
    datos.append({
        "text_a": item.get("text_a", "").strip(),
        "text_b": item.get("text_b", "").strip(),
        "label" : int(label),
    })

print(f"Total pares: {len(datos)}")
print("Distribución de clases:")
dist = Counter(d["label"] for d in datos)
for l, n in sorted(dist.items()):
    print(f"  {ID2LABEL[l]:>18}: {n}")

# Split estratificado 80/10/10
from sklearn.model_selection import train_test_split

labels_all = [d["label"] for d in datos]
train_data, temp = train_test_split(datos, test_size=0.20, stratify=labels_all, random_state=SEED)
labels_temp = [d["label"] for d in temp]
val_data, test_data = train_test_split(temp, test_size=0.50, stratify=labels_temp, random_state=SEED)

print(f"\nSplit: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

Total pares: 2500
Distribución de clases:
       CONTRADICCION: 500
      COMPATIBILIDAD: 500
      ESPECIFICACION: 500
           EXCEPCION: 500
         NO_RELACION: 500

Split: Train=2000, Val=250, Test=250


In [5]:
# Cargar tokenizer y tokenizar
from transformers import AutoTokenizer
from datasets import Dataset

# MODEL_BASE = "cross-encoder/nli-deberta-v3-large"
MODEL_BASE = "cross-encoder/nli-deberta-v3-base"
MAX_LENGTH = 512

print(f"Cargando tokenizer de {MODEL_BASE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)
print("✓ Tokenizer cargado")

def tokenizar(datos_lista, tokenizer, max_len=MAX_LENGTH):
    ds = Dataset.from_dict({
        "text_a": [d["text_a"] for d in datos_lista],
        "text_b": [d["text_b"] for d in datos_lista],
        "label" : [d["label"]  for d in datos_lista],
    })
    def tok(batch):
        return tokenizer(
            batch["text_a"], batch["text_b"],
            truncation=True, max_length=max_len, padding="max_length",
        )
    return ds.map(tok, batched=True, remove_columns=["text_a", "text_b"])

print("Tokenizando...")
train_ds = tokenizar(train_data, tokenizer)
val_ds   = tokenizar(val_data,   tokenizer)
test_ds  = tokenizar(test_data,  tokenizer)
print(f" Train:{len(train_ds)} Val:{len(val_ds)} Test:{len(test_ds)}")

In [7]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
# Cargar modelo y aplicar LoRA
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

print(f"Cargando {MODEL_BASE} (~1.7GB, ~3 min)...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_BASE,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
print(f" Modelo cargado")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,      # clasificación de secuencia
    r=16,                             # rango de las matrices LoRA
                                      # r=8: menos params, menos capacidad
                                      # r=16: buen balance para 2500 muestras
                                      # r=32: más capacidad, más riesgo overfitting
    lora_alpha=32,                    # escalado = alpha/r = 2.0
                                      # convención estándar: alpha = 2*r
    target_modules=[                  # módulos de atención de DeBERTa-v3
        "query_proj",
        "key_proj",
        "value_proj",
        "pos_query_proj",
        "pos_key_proj",
    ],
    lora_dropout=0.1,                 # regularización adicional
    bias="none",                      # no adaptar bias (más eficiente)
)

model = get_peft_model(model, lora_config)

# Verificar cuántos parámetros se entrenan
params_total      = sum(p.numel() for p in model.parameters())
params_trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParámetros totales:     {params_total:>12,}")
print(f"Parámetros entrenables: {params_trainable:>12,} "
      f"({params_trainable/params_total*100:.2f}%)")
print(f"Parámetros congelados:  {params_total-params_trainable:>12,}")
print()
print("Interpretación:")
print(f"  LoRA entrena {params_trainable/1e6:.1f}M parámetros")
print(f"  vs Full FT: {params_total/1e6:.0f}M parámetros")
print(f"  Reducción: {(1-params_trainable/params_total)*100:.1f}% menos parámetros")

In [7]:
# Métricas
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    # F1 macro: trata todas las clases por igual
    # Importante para dataset balanceado (500 por clase)
    f1_macro  = f1_score(labels, preds, average="macro",  zero_division=0)
    f1_contra = f1_score(labels, preds, average=None,     zero_division=0)[0]  # clase 0
    acc       = accuracy_score(labels, preds)
    return {
        "f1_macro"       : round(f1_macro, 4),
        "f1_contradiccion": round(f1_contra, 4),  # la más importante para la tesis
        "accuracy"       : round(acc, 4),
    }

print(" Métricas definidas")
print("  Métrica principal: f1_macro (trata las 5 clases por igual)")
print("  Métrica secundaria: f1_contradiccion (clase 0, la que importa en la tesis)")

In [8]:
# Configurar entrenamiento
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# Con LoRA en T4, podemos usar batch más grande que full FT
BATCH_SIZE  = 16
EPOCHS      = 8
LR          = 2e-4   # LoRA usa LR más alto que full FT (2e-4 vs 2e-5)
                     # porque los parámetros LoRA se inicializan en 0
                     # y necesitan moverse más rápido

OUTPUT_DIR = "./deberta_lora_5clases"

pasos_por_epoca = len(train_ds) // BATCH_SIZE
tiempo_est = pasos_por_epoca * EPOCHS * 0.20 / 60
print(f"Pasos/época: {pasos_por_epoca}")
print(f"Tiempo estimado T4: {tiempo_est:.0f}-{tiempo_est*1.3:.0f} min")

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=20,
    report_to="none",
    seed=SEED,
    fp16=True,         # T4 soporta fp16
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print("✓ Trainer configurado")

In [10]:
# ENTRENAR
import time
print("Iniciando LoRA fine-tuning...")
t0 = time.time()
trainer.train()
print(f"\n Entrenamiento completado en {(time.time()-t0)/60:.1f} minutos")

In [11]:
# Evaluación final sobre test set
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Evaluando sobre test set...")
test_results = trainer.evaluate(test_ds)
print(f"\nResultados en test set:")
print(f"  F1 macro:          {test_results['eval_f1_macro']:.4f}")
print(f"  F1 Contradicción:  {test_results['eval_f1_contradiccion']:.4f}")
print(f"  Accuracy:          {test_results['eval_accuracy']:.4f}")

# Predicciones para reporte detallado
pred_output = trainer.predict(test_ds)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print("\nReporte por clase:")
print(classification_report(
    y_true, y_pred,
    target_names=list(ID2LABEL.values()),
    zero_division=0
))

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=list(ID2LABEL.values()),
    yticklabels=list(ID2LABEL.values())
)
plt.title(f"Matriz de Confusión — DeBERTa-v3-large LoRA\nF1 macro={test_results['eval_f1_macro']:.3f}")
plt.xlabel("Predicho"); plt.ylabel("Real")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("matriz_confusion_5clases.png", dpi=150)
plt.show()
print("✓ Matriz guardada")

In [17]:
# Verificar inferencia del modelo subido
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print(f"Cargando modelo mergeado desde HuggingFace: {HF_REPO}")
tok_v = AutoTokenizer.from_pretrained(HF_REPO)
mod_v = AutoModelForSequenceClassification.from_pretrained(HF_REPO)
mod_v.eval()

ID2LABEL_V = mod_v.config.id2label
print(f"id2label: {ID2LABEL_V}")

def predecir_relacion(text_a, text_b):
    inputs = tok_v(
        text_a, text_b,
        return_tensors="pt", truncation=True,
        max_length=512, padding=True
    )
    with torch.no_grad():
        probs = torch.softmax(mod_v(**inputs).logits, dim=1)[0]
    idx_pred = probs.argmax().item()
    return {
        "etiqueta": ID2LABEL_V[idx_pred],
        "confianza": round(float(probs[idx_pred]), 4),
        "scores": {ID2LABEL_V[i]: round(float(probs[i]), 4) for i in range(len(probs))},
    }

# Tests con ejemplos de cada clase
tests = [
    ("El arrendatario debe usar el bien con diligencia ordinaria.",
     "El arrendatario puede subarrendar el bien sin restricción.",
     "CONTRADICCION"),
    ("El domicilio se constituye por residencia habitual.",
     "La persona que vive en varios lugares se considera domiciliada en cualquiera.",
     "COMPATIBILIDAD"),
    ("La persona humana es sujeto de derecho desde el nacimiento.",
     "El concebido es sujeto de derecho para todo cuanto le favorece.",
     "ESPECIFICACION"),
    ("Las partidas de inscripción prueban los hechos.",
     "Pueden hacerse rectificaciones en las partidas por resolución judicial.",
     "EXCEPCION"),
    ("La persona humana es sujeto de derecho desde el nacimiento.",
     "El interés moratorio se genera por el incumplimiento de una obligación.",
     "NO_RELACION"),
]

print("\nTests de verificación:")
correctos = 0
for ta, tb, esperado in tests:
    r = predecir_relacion(ta, tb)
    ok = "ok" if r["etiqueta"] == esperado else "fail"
    print(f"  {ok} Esperado={esperado:>15} | Predicho={r['etiqueta']:>15} (conf={r['confianza']:.3f})")
    if r["etiqueta"] == esperado:
        correctos += 1

print(f"\nAciertos verificación: {correctos}/5")
print(f"\n Modelo listo para usar en el notebook del ablation study")
print(f"  FINETUNED_MODEL = '{HF_REPO}'")

Cargando modelo mergeado desde HuggingFace: sinai1carlos/deberta-lora-nli-juridico-peru-5clases


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

id2label: {0: 'CONTRADICCION', 1: 'COMPATIBILIDAD', 2: 'ESPECIFICACION', 3: 'EXCEPCION', 4: 'NO_RELACION'}

Tests de verificación:
  ok Esperado=  CONTRADICCION | Predicho=  CONTRADICCION (conf=0.995)
  ok Esperado= COMPATIBILIDAD | Predicho= COMPATIBILIDAD (conf=0.444)
  ok Esperado= ESPECIFICACION | Predicho= ESPECIFICACION (conf=0.808)
  ok Esperado=      EXCEPCION | Predicho=      EXCEPCION (conf=0.961)
  ok Esperado=    NO_RELACION | Predicho=    NO_RELACION (conf=0.780)

Aciertos verificación: 5/5

 Modelo listo para usar en el notebook del ablation study
  FINETUNED_MODEL = 'sinai1carlos/deberta-lora-nli-juridico-peru-5clases'
